# Stock Market Data Warehouse — ETL Pipeline

This notebook walks through the full **Extract → Transform → Load** pipeline that builds the star-schema data warehouse.

## Star Schema
```
          dim_company
               │
dim_date ─── fact_stock_prices ─── dim_financials
```

| Table | Grain | Rows (approx.) |
|---|---|---|
| `dim_company` | 1 row / symbol | 9 |
| `dim_date` | 1 row / trading date | ~10,000 |
| `dim_financials` | 1 row / (symbol, year) | 72 |
| `fact_stock_prices` | 1 row / (date, symbol) | ~50,000 |

**Symbols tracked:** AAPL, MSFT, GOOGL, AMZN, TSLA, META, NFLX, NVDA, INTC

In [ ]:
import sys, os
# Allow importing the etl package from the repo root
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

DATA_DIR = Path('../data')
OUT_DIR  = Path('../outputs')
OUT_DIR.mkdir(exist_ok=True)

print('Setup complete.')

## 1. EXTRACT — Load Raw Data

In [ ]:
from etl.extract import extract_stock_prices, extract_company_metadata, extract_financials

raw_prices     = extract_stock_prices(use_cache=True)
raw_company    = extract_company_metadata()
raw_financials = extract_financials(use_cache=True)

print(f'Stock price rows  : {len(raw_prices):,}')
print(f'Company rows      : {len(raw_company)}')
print(f'Financials rows   : {len(raw_financials)}')

In [ ]:
raw_prices.head()

In [ ]:
raw_company

In [ ]:
raw_financials.head()

## 2. TRANSFORM — Build Star Schema Dimensions

In [ ]:
from etl.transform import (
    transform_dim_company,
    transform_dim_date,
    transform_dim_financials,
    transform_fact_stock_prices,
    validate_star_schema,
)

dim_company    = transform_dim_company(raw_company)
fact_prices    = transform_fact_stock_prices(raw_prices)
dim_date       = transform_dim_date(fact_prices)
dim_financials = transform_dim_financials(raw_financials)

print('Transformation complete.')

In [ ]:
# dim_company
print('=== dim_company ===')
print(dim_company.dtypes)
dim_company

In [ ]:
# dim_date — first and last few rows
print(f'dim_date: {len(dim_date):,} rows | {dim_date.Date.min().date()} → {dim_date.Date.max().date()}')
dim_date.head()

In [ ]:
# fact_stock_prices summary
print(f'fact_stock_prices: {len(fact_prices):,} rows')
print(f'Symbols : {sorted(fact_prices.Symbol.unique())}')
print(f'Date range: {fact_prices.Date.min().date()} → {fact_prices.Date.max().date()}')
fact_prices.describe()

In [ ]:
# dim_financials summary
print(f'dim_financials: {len(dim_financials)} rows')
dim_financials[['Symbol','Year','Revenue','Net_Income','ROE','ROA']].sort_values(['Symbol','Year']).head(15)

### 2a. Data Quality Checks

In [ ]:
# Null check — fact table
nulls = fact_prices.isnull().sum()
print('Null counts in fact_stock_prices:')
print(nulls[nulls > 0] if nulls.sum() > 0 else '  None — clean!')

# Duplicate check
dupes = fact_prices.duplicated(subset=['Date','Symbol']).sum()
print(f'\nDuplicate (Date, Symbol) rows: {dupes}')

In [ ]:
# Star-schema referential integrity validation
validation = validate_star_schema(fact_prices, dim_company, dim_date)
for k, v in validation.items():
    print(f'  {k}: {v}')
print('\nValidation passed!' if validation['valid'] else '\nValidation FAILED — check above.')

In [ ]:
# Rows per symbol
fact_prices.groupby('Symbol').agg(
    rows=('Close','count'),
    date_min=('Date','min'),
    date_max=('Date','max'),
    avg_close=('Close','mean'),
).round(2)

## 3. LOAD — Export to CSV (and optionally PostgreSQL)

In [ ]:
from etl.load import export_to_csv

export_to_csv(dim_company, dim_date, dim_financials, fact_prices, out_dir=DATA_DIR)
print('CSVs written to data/')

In [ ]:
# Optional: load into PostgreSQL
# Uncomment and run `docker-compose up -d` first

# import os
# from dotenv import load_dotenv
# load_dotenv('../.env')
# from etl.load import load_to_postgres
# load_to_postgres(dim_company, dim_date, dim_financials, fact_prices)

## 4. Quick Exploratory Plots

In [ ]:
# Merge for analysis
df = fact_prices.merge(dim_date, on='Date').merge(
    dim_company[['Symbol','Sector','Name','Beta']], on='Symbol'
)

In [ ]:
# Closing price over time — all symbols
fig, ax = plt.subplots(figsize=(14, 6))
for sym, grp in fact_prices.groupby('Symbol'):
    ax.plot(grp['Date'], grp['Close'], linewidth=0.8, label=sym)
ax.set_title('Daily Closing Price — All Symbols', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Close Price (USD)')
ax.legend(ncol=3, fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig(OUT_DIR / 'closing_price_all.png', dpi=150)
plt.show()

In [ ]:
# Row count per symbol (data availability)
counts = fact_prices.groupby('Symbol')['Close'].count().sort_values()
fig, ax = plt.subplots(figsize=(8,4))
counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Trading Days')
ax.set_title('Data Availability per Symbol', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'data_availability.png', dpi=150)
plt.show()

In [ ]:
print('ETL notebook complete. Proceed to 02_analysis.ipynb for full analysis.')